# Cryptocurrency Market Data Collection

This notebook builds a reproducible daily closing-price dataset for
USDT-denominated spot cryptocurrencies.

The project was inspired by Chee-Foong's cryptocurrency data-extraction
notebook and Peter Nistrup's historical Binance-data article. The
implementation has been restructured and extended for this research project.

## References

- Chee-Foong, *Pairs Trading Using Co-integrated Cryptocurrency Pairs*,
  GitHub, 2020.
- Peter Nistrup, *Retrieving Full Historical Data for Every Cryptocurrency
  on Binance & BitMEX Using the Python APIs*, 2019.

## Install dependencies

In [1]:
!pip install python-binance


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Loading the libraries

In [2]:
# IMPORTS
import pandas as pd
import numpy as np

import time
import math
import os.path

from tqdm import tqdm_notebook #(Optional, used for progress-bars)
from datetime import timedelta, datetime
from dateutil import parser

import os
import math
import pandas as pd
from datetime import datetime
from dateutil import parser
from binance.client import Client
from tqdm import tqdm

In [3]:
from binance.client import Client

### API
binance_api_key = api['key']       #Enter your own API-key here
binance_api_secret = api['secret'] #Enter your own API-secret here

### CONSTANTS
binsizes = {"1d": 1440}
binance_client = Client(api_key=binance_api_key, api_secret=binance_api_secret)

# Set the path name for the data folder and create it if it doesn't exist
data_folder = 'Desktop/data/'
if not os.path.exists(data_folder):
    os.makedirs(data_folder)

NameError: name 'api' is not defined

In [ ]:
# Define the fixed start and end dates for data collection
START_DATE = datetime.strptime('1 Jan 2018', '%d %b %Y')
END_DATE = datetime.strptime('31 Dec 2024', '%d %b %Y')

def minutes_of_new_data(symbol, kline_size, data, source):
    """
    Determines the start (old) and end (new) dates for downloading data.
    If previous data exists, it resumes from the last timestamp;
    otherwise it uses the fixed START_DATE.
    """
    if len(data) > 0:
        # Use the last timestamp in the stored data
        old = parser.parse(data.index[-1])
    elif source == "binance":
        old = START_DATE
    # Force the new endpoint to the fixed END_DATE
    new = END_DATE
    return old, new

def get_all_binance(symbol, kline_size, save=False):
    """
    Downloads historical klines from Binance for a given symbol and interval.
    It keeps only the timestamp and closing price, then stores the data into a CSV.
    """
    filename = os.path.join(data_folder, f'{symbol}-{kline_size}-data.csv')
    
    if os.path.isfile(filename):
        # Load existing data; parse dates and set the timestamp as the index
        data_df = pd.read_csv(filename, index_col='timestamp', parse_dates=True)
    else:
        data_df = pd.DataFrame()
    
    oldest_point, newest_point = minutes_of_new_data(symbol, kline_size, data_df, source="binance")
    delta_days = (newest_point - oldest_point).days
    available_data = delta_days  # Each day represents one data point
    
    if oldest_point == START_DATE:
        print(f'Downloading all available {kline_size} data for {symbol} from '
              f'{oldest_point.strftime("%d %b %Y")} to {newest_point.strftime("%d %b %Y")}. Be patient..!')
    else:
        print(f'Downloading {available_data} days of new data for {symbol} '
              f'({available_data} instances of {kline_size} data).')
    
    klines = binance_client.get_historical_klines(
        symbol, kline_size,
        oldest_point.strftime("%d %b %Y %H:%M:%S"),
        newest_point.strftime("%d %b %Y %H:%M:%S")
    )
    
    # Create a DataFrame from the downloaded data
    data = pd.DataFrame(klines, columns=[
        'timestamp', 'open', 'high', 'low', 'close', 
        'volume', 'close_time', 'quote_av', 'trades', 
        'tb_base_av', 'tb_quote_av', 'ignore'
    ])
    data['timestamp'] = pd.to_datetime(data['timestamp'], unit='ms')
    
    # Keep only the closing price along with the timestamp
    data = data[['timestamp', 'close']]
    data['close'] = pd.to_numeric(data['close'])
    
    # Append new data to existing data (if any) and remove duplicate timestamps
    if not data_df.empty:
        temp_df = data.set_index('timestamp')
        data_df = data_df.append(temp_df)
        data_df = data_df[~data_df.index.duplicated(keep='last')]
    else:
        data_df = data.set_index('timestamp')
    
    if save:
        data_df.to_csv(filename)
        
    print(f'All caught up for {symbol}!')
    return data_df

## Get list of coins

Get a cryptocurrency symbols based in USD

In [ ]:
# Get list of coins – filtering symbols that are paired with USDT
info = binance_client.get_exchange_info()
symbols = info['symbols']
coins = []
others = []

for sym in symbols:
    s = sym['symbol']
    # Assuming primary coins have a symbol length of 7 (e.g., BTCUSDT)
    if 'USDT' in s and len(s) == 7:
        coins.append(s)
    elif 'USDT' in s:
        others.append(s)

In [ ]:
# others

In [ ]:
# Download daily closing prices for each coin in the list
for symbol in tqdm(coins):
    try:
        get_all_binance(symbol, '1d', save=True)
    except Exception as e:
        print(f'Skipping {symbol} due to error: {e}')
        continue

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

data_list = []

# Loop through each coin symbol
for symbol in tqdm(coins):
    filename = os.path.join(data_folder, f'{symbol}-1d-data.csv')
    try:
        df = pd.read_csv(filename, parse_dates=True, index_col='timestamp')
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        continue

    # Select only the 'close' column (already daily data)
    df = df[['close']]
    # Rename the column using the coin's base name (e.g. 'BTCUSDT' becomes 'BTC')
    col_name = symbol.replace("USDT", "")
    df.rename(columns={'close': col_name}, inplace=True)
    data_list.append(df)

# Concatenate all DataFrames on the timestamp index (axis=1)
prices = pd.concat(data_list, axis=1)

# Save the merged DataFrame to a CSV file on the Desktop
output_file = os.path.join(os.path.expanduser("~"), "Desktop", "Book4.csv")
prices.to_csv(output_file)
print("Merged data saved to:", output_file)


Saving the prices in a csv file for future analysis

In [ ]:
prices.to_csv('Desktop/Book4.csv')

Reloading prices for checking

In [ ]:
prices = pd.read_csv('Desktop/Book4.csv', parse_dates=True, index_col='timestamp')
prices.head()